# Google Geocoding CSV ツール (Colab)\n\nこのノートブックは `geocode_csv.py` を Google Colab 上で実行するためのものです。

## 1) API キーを設定

In [ ]:
import os
import getpass

os.environ['GOOGLE_MAPS_API_KEY'] = getpass.getpass('Google Maps API Key: ')
print('API キーを環境変数 GOOGLE_MAPS_API_KEY に設定しました。')

## 2) 入力CSVをアップロード

In [ ]:
from google.colab import files

uploaded = files.upload()
print('アップロード済みファイル:', list(uploaded.keys()))

## 3) `geocode_csv.py` を利用する準備
このノートブックではロジックを持たず、`geocode_csv.py` を実行します。
- `geocode_csv.py` がまだ Colab 上にない場合は、入力CSVと一緒にアップロードしてください。


In [ ]:
from pathlib import Path

SCRIPT_PATH = Path('geocode_csv.py')
if not SCRIPT_PATH.exists():
    raise FileNotFoundError(
        'geocode_csv.py が見つかりません。CSVと一緒に geocode_csv.py をアップロードしてください。'
    )

print(f'実行スクリプト: {SCRIPT_PATH.resolve()}')


## 4) 変換を実行（ログを時刻付きファイルに保存）
ログは `logs/` 配下に `geocode_YYYYMMDD_HHMMSS.log` 形式で保存します（上書きしません）。


In [ ]:
# 必要に応じてファイル名を変更してください
INPUT_CSV = 'サンプル_飲食店_豊中_緯度経度なし.csv'
OUTPUT_CSV = 'output_with_latlng.csv'

# 任意オプション
DISABLE_PLACES = False
PLACES_BIAS_RADIUS = 3000
SLEEP_SECONDS = 0.1
TIMEOUT_SECONDS = 10


In [ ]:
import os
import subprocess
from datetime import datetime
from pathlib import Path

API_KEY = os.environ.get('GOOGLE_MAPS_API_KEY', '').strip()
if not API_KEY:
    raise ValueError('GOOGLE_MAPS_API_KEY が未設定です。最初のセルを実行してください。')

logs_dir = Path('logs')
logs_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = logs_dir / f'geocode_{timestamp}.log'

cmd = [
    'python',
    str(SCRIPT_PATH),
    INPUT_CSV,
    OUTPUT_CSV,
    '--places-bias-radius', str(PLACES_BIAS_RADIUS),
    '--sleep', str(SLEEP_SECONDS),
    '--timeout', str(TIMEOUT_SECONDS),
]
if DISABLE_PLACES:
    cmd.append('--disable-places')

print('実行コマンド:', ' '.join(cmd))
print('ログファイル:', log_path)

with log_path.open('w', encoding='utf-8') as logf:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        logf.write(line)

return_code = proc.wait()
print(f'終了コード: {return_code}')
if return_code != 0:
    raise RuntimeError(f'geocode_csv.py の実行に失敗しました。ログを確認してください: {log_path}')

print(f'完了。ログ保存先: {log_path}')


## 5) 出力CSVをダウンロード
必要なら `logs/` 配下のログファイルもダウンロードしてください。


In [ ]:
files.download(OUTPUT_CSV)
